In [ ]:
!pip install onnxruntime

# YOLO

In [ ]:
!pip install ultralytics

In [ ]:
from ultralytics import YOLO

model = YOLO("models/yolov8l.pt")
results = model.predict(source="city.jpg" , save=True)
# for result in results :
#     result.save(filename="yolo_output.jpg")

model.export(format="onnx")


image 1/1 f:\pyprogram\pydeploy\PyDeploy\onnx_runtime\city.jpg: 640x448 12 persons, 3 cars, 2 buss, 2 traffic lights, 1 tie, 636.3ms
Speed: 2.0ms preprocess, 636.3ms inference, 2.4ms postprocess per image at shape (1, 3, 640, 448)
Results saved to runs\detect\predict2
Ultralytics YOLOv8.0.220 🚀 Python-3.10.1 torch-2.1.1+cpu CPU (11th Gen Intel Core(TM) i5-11300H 3.10GHz)

PyTorch: starting from 'models\yolov8l.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 84, 8400) (83.7 MB)

ONNX: starting export with onnx 1.16.1 opset 17...
ONNX: export success ✅ 3.2s, saved as 'models\yolov8l.onnx' (166.8 MB)

Export complete (6.6s)
Results saved to F:\pyprogram\pydeploy\PyDeploy\onnx_runtime\models
Predict:         yolo predict task=detect model=models\yolov8l.onnx imgsz=640  
Validate:        yolo val task=detect model=models\yolov8l.onnx imgsz=640 data=coco.yaml  
Visualize:       https://netron.app


'models\\yolov8l.onnx'

In [ ]:
# INFERENCE
# load model :
onnx_model = YOLO("models/yolov8l.onnx" , task='detect')

# use model :
result = onnx_model("city.jpg")
result

Loading models\yolov8l.onnx for ONNX Runtime inference...

image 1/1 f:\pyprogram\pydeploy\PyDeploy\onnx_runtime\city.jpg: 640x640 12 persons, 3 cars, 2 buss, 2 traffic lights, 1 tie, 728.6ms
Speed: 2.9ms preprocess, 728.6ms inference, 14.1ms postprocess per image at shape (1, 3, 640, 640)


[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: None
 masks: None
 names: {0: 'person', 1: 'bicycle', 2: 'car', 3: 'motorcycle', 4: 'airplane', 5: 'bus', 6: 'train', 7: 'truck', 8: 'boat', 9: 'traffic light', 10: 'fire hydrant', 11: 'stop sign', 12: 'parking meter', 13: 'bench', 14: 'bird', 15: 'cat', 16: 'dog', 17: 'horse', 18: 'sheep', 19: 'cow', 20: 'elephant', 21: 'bear', 22: 'zebra', 23: 'giraffe', 24: 'backpack', 25: 'umbrella', 26: 'handbag', 27: 'tie', 28: 'suitcase', 29: 'frisbee', 30: 'skis', 31: 'snowboard', 32: 'sports ball', 33: 'kite', 34: 'baseball bat', 35: 'baseball glove', 36: 'skateboard', 37: 'surfboard', 38: 'tennis racket', 39: 'bottle', 40: 'wine glass', 41: 'cup', 42: 'fork', 43: 'knife', 44: 'spoon', 45: 'bowl', 46: 'banana', 47: 'apple', 48: 'sandwich', 49: 'orange', 50: 'broccoli', 51: 'carrot', 52: 'hot dog', 53: 'pizza', 54: 'donut', 55: 'cake', 56: 'chair', 57: 'couch', 58: 'potted p

# Scikit-Learn

In [ ]:
!pip install skl2onnx

In [ ]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
import skl2onnx
import onnx

iris = load_iris()
X , Y = iris.data , iris.target
X = X.astype(np.float32)
X_train , X_test , Y_train , Y_test = train_test_split(X , Y)

model = RandomForestClassifier()
model.fit(X_train , Y_train)

onnx_model = skl2onnx.to_onnx(model , X[:1])
with open("models/sklearn_2_onnx.onnx" , mode="wb") as modelfile :
    modelfile.write(onnx_model.SerializeToString())

# onnx model created

### INFERENCE : USE  & LOAD model
if we open model file in Netron , in output we can see we have 2 outputs :
<br>
+ 1_ output_probability
<br>
+ 2_ output_label

In [ ]:
X[:1]

array([[5.1, 3.5, 1.4, 0.2]], dtype=float32)

In [ ]:
X[0]

array([5.1, 3.5, 1.4, 0.2], dtype=float32)

In [ ]:
X_test[:1]

array([[6.3, 3.3, 6. , 2.5]], dtype=float32)

In [ ]:
# use onnx model
import onnxruntime
import onnx
import numpy as np


model = onnx.load('models/sklearn_2_onnx.onnx')
model.ir_version = 9
onnx.save(model, 'models/sklearn_2_onnx.onnx')


onnx_model = onnxruntime.InferenceSession("models/sklearn_2_onnx.onnx" , providers=["CPUExecutionProvider"])

for input_meta in onnx_model.get_inputs():
    print(f"Input name: {input_meta.name}, shape: {input_meta.shape}, type: {input_meta.type}")


input_name = onnx_model.get_inputs()[0].name
print(input_name)
output_name = onnx_model.get_outputs()[0].name
print(output_name)

#use onnx model
# prediction = onnx_model.run(output_names=["output_label" or "output_probability"] , input_feed={"X" : X_test[0] } )
prediction = onnx_model.run(output_names=[output_name] , input_feed={input_name : X_test.astype(np.float32) } )[0] # output_names= LIST , input_name= DICTIONARY{ input_name : input_value }
prediction

Input name: X, shape: [None, 4], type: tensor(float)
X
output_label


array([2, 2, 2, 0, 1, 1, 0, 1, 1, 1, 2, 0, 2, 1, 2, 0, 2, 0, 0, 1, 0, 2,
       1, 1, 0, 2, 0, 1, 1, 2, 2, 0, 2, 0, 0, 1, 1, 2], dtype=int64)

## Scikit-Learn with Thread management :

+ By default optimum with onnxruntime will use all available cores. If you want to adjust this, e.g. use multiprocessing and optimum in parallel you can adjust the used THREADS/CPU cores with onnxruntime.SessionOptions by configuring inter_op_num_threads and intra_op_num_threads

In [ ]:
import time

onnx_model_path = "models/sklearn_2_onnx.onnx"
test_input = np.array(X, dtype=np.float32)
Total_INTRA_Threads = 5 #Number of physical CPU Cores


# SEQUENTIAL
sess_options = onnxruntime.SessionOptions()
# By default intra_op=0 or not set, each session will start with the main thread on the 1st core.
# Then extra threads per additional physical core are created, and affinitized to that core.
# intra_op = 3 total threads ==  2 extra threads + 1 INTRA pool (main calling thread)
sess_options.intra_op_num_threads = 0
sess_options.execution_mode = onnxruntime.ExecutionMode.ORT_SEQUENTIAL
sess_options.graph_optimization_level = onnxruntime.GraphOptimizationLevel.ORT_ENABLE_ALL
sess_options.add_session_config_entry("session.intra_op.allow_spinning", "1")
session = onnxruntime.InferenceSession(onnx_model_path, sess_options=sess_options , providers=['CPUExecutionProvider'])
input_name = session.get_inputs()[0].name
output_name = session.get_outputs()[0].name
start_time = time.time()
predictions = session.run([output_name], {input_name: test_input})[0]
end_time = time.time()
sequential_duration = end_time - start_time
print(f"Inference time with {1} threads: {sequential_duration:.6f} seconds\n")
print("---------------------------------------------------------------------")




# Set intra-op thread affinity :
# There are multiple sessions run in parallel, customer might prefer their intra-op thread pools run on separate cores to avoid contention
sess_opt = onnxruntime.SessionOptions()
sess_opt.intra_op_num_threads = 3
sess_opt.add_session_config_entry('session.intra_op_thread_affinities', '1;2')
sess = onnxruntime.InferenceSession(onnx_model_path , sess_opt, providers=['CPUExecutionProvider'])
input_name2 = sess.get_inputs()[0].name
output_name2 = sess.get_outputs()[0].name
start_time2 = time.time()
predictions2 = session.run([output_name2], {input_name2: test_input})[0]
end_time2 = time.time()
sequential_duration2 = end_time2 - start_time2
print(f"Inference time with {sess_opt.intra_op_num_threads} threads: {sequential_duration2:.6f} seconds\n")
print("---------------------------------------------------------------------")


# -------------------PARALLEL-----------------------
# when a model has many branches --> better performance
for num_threads in [1,2,3,4,5,6,7,8]:  # numbers of threads
    session_options = onnxruntime.SessionOptions()
    session_options.execution_mode = onnxruntime.ExecutionMode.ORT_PARALLEL
    session_options.intra_op_num_threads = Total_INTRA_Threads  # Numbers of threads for parallelism
    # inter-op thread pool is for parallelism between operators, and will only be created when session execution mode set to parallel
    # By default, inter-op thread pool will also have one thread per physical core --> default=5
    session_options.inter_op_num_threads = num_threads # to control the number of threads used to parallelize the execution

    session = onnxruntime.InferenceSession(onnx_model_path, sess_options=session_options , providers=['CUDAExecutionProvider'])
    input_name = session.get_inputs()[0].name
    output_name = session.get_outputs()[0].name

    start_time = time.time()
    predictions = session.run([output_name], {input_name: test_input})[0]
    end_time = time.time()
    parallel_duration = end_time - start_time
    print(f"Inference time with {num_threads} threads: {parallel_duration:.6f} seconds")


Inference time with 1 threads: 0.002003 seconds

---------------------------------------------------------------------
Inference time with 3 threads: 0.000000 seconds

---------------------------------------------------------------------
Inference time with 1 threads: 0.000999 seconds
Inference time with 2 threads: 0.000416 seconds
Inference time with 3 threads: 0.005470 seconds
Inference time with 4 threads: 0.000499 seconds
Inference time with 5 threads: 0.000000 seconds
Inference time with 6 threads: 0.001001 seconds
Inference time with 7 threads: 0.000498 seconds
Inference time with 8 threads: 0.010151 seconds


+ ### 1.  the total number of threads do not exceed the number of physical cores in a machine
+ ### 2. Parallel execution of operators is scheduled on an inter-op thread pool. The execution of an individual operator is parallelized using an intra-op thread pool.
+ ### 3.  + intra_op = 3 total threads ==  2 extra threads + 1 INTRA pool (main calling thread)
+ 4. Keeping tensor data on GPU can avoid unnecessary data transfer between CPU and GPU, which can improve the performance.

In [ ]:
# use to a single thread only -
# It's for the default CPU execution provider.
'''
opts.intra_op_num_threads = 1
opts.inter_op_num_threads = 1
opts.execution_mode = onnxruntime.ExecutionMode.ORT_SEQUENTIAL
'''

# Pytorch

www.pytorch.org/docs/stable/generated/torch.nn.functional.max_pool2d.html#torch.nn.functional.max_pool2d

In [ ]:
!pip install torch
!pip install onnx
!pip install onnxscript

In [ ]:
import torch
import onnx
import onnxruntime

class MyModel(torch.nn.Module):
    def __init__(self):
        super(MyModel  , self).__init__()
        self.conv1 = torch.nn.Conv2d(1, 6 ,5)
        self.conv2 = torch.nn.Conv2d(6 ,16 ,5)
        self.fc1 = torch.nn.Linear(16*5*5 , 120 )
        self.fc2 = torch.nn.Linear(120 , 84)
        self.fc3 = torch.nn.Linear(84 ,10)

    def forward(self , x):
        x =  torch.nn.functional.max_pool2d(input= torch.nn.functional.relu(self.conv1(x)) , kernel_size=(2,2) )
        x =  torch.nn.functional.max_pool2d(input= torch.nn.functional.relu(self.conv2(x)) , kernel_size=2 )
        x =  torch.flatten(input=x , start_dim=1)
        x =  torch.nn.functional.relu(input=self.fc1(x))
        x =  torch.nn.functional.relu(input=self.fc2(x))
        x =  self.fc3(x)
        return x


# convert TORCH to ONNX :
torch_model = MyModel()
sample_torch_input = torch.randn(1,1,32,32) # batchsize , channel , imagesize
print("sample_torch_input :" , sample_torch_input)
onnx_model = torch.onnx.export( torch_model , sample_torch_input , "models/torch2onnxmodel.onnx") # model + one_sample_input
# check model :
onnx_model = onnx.load("models/torch2onnxmodel.onnx")
onnx.checker.check_model(onnx_model)

# INFERENCE
# load onnx model :
onnxMODEL = onnxruntime.InferenceSession("models/torch2onnxmodel.onnx" , providers=["CPUExecutionProvider"])
for input_meta in onnxMODEL.get_inputs():
    print(f"Input name: {input_meta.name}, shape: {input_meta.shape}, type: {input_meta.type}")
for output_meta in onnxMODEL.get_outputs():
    print(f"Output name: {output_meta.name}, shape: {output_meta.shape}, type: {output_meta.type}")

# use onnx model :
input_layer_name  = onnxMODEL.get_inputs()[0].name
output_layer_name = onnxMODEL.get_outputs()[0].name
print(input_layer_name)
print(output_layer_name)# check these names with NETRON
# sample_input is a TENSOR , but we should give a numpyarray as dict value
# TENSOR --> numpyarray
# onnxruntime_input = onnx_model.adapt_torch_inputs_to_onnx( sample_torch_input )
onnxruntime_input = sample_torch_input.detach().cpu().numpy()
print("onnxruntime_input : " , onnxruntime_input.dtype)
print(onnxruntime_input)
prediction = onnxMODEL.run([output_layer_name] ,  {input_layer_name : onnxruntime_input })
print("prediction :" , prediction)

sample_torch_input : tensor([[[[-1.1838e+00, -2.1124e+00, -1.0001e-03,  ..., -1.2644e+00,
            4.1426e-01,  5.2796e-02],
          [ 1.9238e+00,  1.5018e-01, -1.1963e-01,  ...,  6.7015e-02,
            2.8714e-02,  8.7087e-01],
          [ 1.5105e+00, -7.7473e-01, -6.1879e-01,  ...,  7.1968e-01,
           -6.2472e-01,  6.9382e-01],
          ...,
          [-9.1262e-01, -9.5170e-01, -6.9433e-02,  ..., -1.2920e+00,
           -3.5789e-01, -4.0522e-01],
          [ 6.1922e-01, -8.8201e-01,  1.6848e+00,  ...,  3.6063e-01,
           -1.4337e+00, -2.1688e+00],
          [-3.1883e-01,  1.3118e+00,  7.6085e-01,  ...,  9.4871e-01,
           -1.5852e+00, -2.5112e-01]]]])
Input name: input.1, shape: [1, 1, 32, 32], type: tensor(float)
Output name: 22, shape: [1, 10], type: tensor(float)
input.1
22
onnxruntime_input :  float32
[[[[-1.1837668e+00 -2.1124234e+00 -1.0001307e-03 ... -1.2644386e+00
     4.1426301e-01  5.2796286e-02]
   [ 1.9238491e+00  1.5017828e-01 -1.1962742e-01 ...  6.701

### compare pytorch result with onnxruntime result  :

In [ ]:

torch_outputs = torch_model(sample_torch_input)
print("torch_outputs :  " , torch_outputs)
print("\ntorch_outputs size :  " , torch_outputs.size())

print("_____________________________________________")
print("prediction :    " , prediction )
print("\nprediction size:    " , len(prediction ))

print("_____________________________________________")
torch_outputs = torch_outputs.unsqueeze(0)
print("unsqueezed: " , torch_outputs)
print(torch_outputs.size())

print("_____________________________________________")
x = torch_outputs.clone().detach().cpu().numpy().astype(np.float32)
print("converted to numpy : " , x)
print("\ntorch_outputs size : " , len(x))

print("_____________________________________________")

assert len(x) == len(prediction)
for torch_output, onnxruntime_output in zip(torch_outputs, prediction):
    torch.testing.assert_close(torch_output, torch.tensor(onnxruntime_output)) # assert_close :  means (torch_outputs) & (prediction) should be similar to each other (they don't need to be exactly the same)

print("\nPyTorch and ONNX Runtime output matched!")
print(f"Output length: {len(prediction)}")
print(f"Sample output: {prediction}")

torch_outputs :   tensor([[ 0.0076,  0.0951, -0.0941,  0.0887, -0.1145, -0.0145, -0.0628, -0.0578,
         -0.1807,  0.0683]], grad_fn=<AddmmBackward0>)

torch_outputs size :   torch.Size([1, 10])
_____________________________________________
prediction :     [array([[ 0.00756476,  0.09511547, -0.09406482,  0.0887133 , -0.11449014,
        -0.01449363, -0.06277543, -0.05777196, -0.18070483,  0.06830081]],
      dtype=float32)]

prediction size:     1
_____________________________________________
unsqueezed:  tensor([[[ 0.0076,  0.0951, -0.0941,  0.0887, -0.1145, -0.0145, -0.0628,
          -0.0578, -0.1807,  0.0683]]], grad_fn=<UnsqueezeBackward0>)
torch.Size([1, 1, 10])
_____________________________________________
converted to numpy :  [[[ 0.00756475  0.09511546 -0.09406481  0.08871327 -0.11449013
   -0.01449364 -0.06277543 -0.05777196 -0.1807048   0.06830081]]]

torch_outputs size :  1
_____________________________________________

PyTorch and ONNX Runtime output matched!
Output le

# Tensorflow

In [ ]:
!pip install tf2onnx

In [13]:
import numpy as np
from keras.applications.resnet50 import ResNet50
import tf2onnx
import tensorflow as tf
import keras
import onnx

model = keras.applications.resnet50.ResNet50(weights="imagenet" , include_top=True )
onnx_model  , x = tf2onnx.convert.from_keras(model , [tf.TensorSpec(shape=model.inputs[0].shape , dtype=model.inputs[0].dtype  , name=model.inputs[0].name )] ) # should pass model & input-sample

onnx.save(onnx_model , "/content/tensorflow2onnx2.onnx")

In [ ]:
import onnxruntime

# load onnx model
onnxMODEL = onnxruntime.InferenceSession("models/tensorflow2onnx.onnx" , providers=["CPUExecutionProvider"] )
input_layer_name  = onnxMODEL.get_inputs()[0].name
output_layer_name = onnxMODEL.get_outputs()[0].name
print(input_layer_name)
print(output_layer_name)

# use onnx model

onnxinput  = np.random.normal(size=[1,224,224,3]).astype(np.float32)
start_time = time.time()
prediction = onnxMODEL.run(output_names= [output_layer_name], input_feed={input_layer_name : onnxinput })

end_time = time.time()
duration = end_time - start_time
print(f"Inference time with CPU : {duration} seconds")
# prediction
# imagenet has 1000 numbers of class , so we have 1000 prediction


keras_tensor
predictions
Inference time with CPU : 0.08394360542297363 seconds


In [ ]:
from PIL import Image

test_img = Image.open("city.jpg")
test_img = test_img.resize((224,224)) # bc resnet images are (224,224)
test_img = np.array(test_img).astype(np.float32)
test_img = np.expand_dims(test_img , axis=0)

onnxinput = test_img
onnxinput

array([[[[227., 234., 241.],
         [227., 234., 241.],
         [227., 234., 243.],
         ...,
         [255., 255., 255.],
         [255., 255., 255.],
         [255., 255., 255.]],

        [[227., 234., 239.],
         [227., 234., 239.],
         [227., 234., 240.],
         ...,
         [239., 244., 248.],
         [239., 244., 247.],
         [240., 245., 247.]],

        [[228., 235., 241.],
         [227., 234., 240.],
         [227., 234., 240.],
         ...,
         [184., 201., 228.],
         [184., 201., 228.],
         [185., 202., 228.]],

        ...,

        [[ 86., 103., 123.],
         [ 81.,  98., 118.],
         [ 74.,  91., 112.],
         ...,
         [132., 144., 168.],
         [129., 141., 166.],
         [129., 142., 168.]],

        [[ 77.,  94., 112.],
         [ 79.,  96., 114.],
         [ 73.,  89., 106.],
         ...,
         [128., 140., 164.],
         [133., 145., 170.],
         [132., 145., 170.]],

        [[ 84., 101., 121.],
       

In [ ]:
from keras.applications.resnet50 import decode_predictions

decode_predictions(prediction[0])


### Tensorflow with ONNXRUNTIME-GPU

In [17]:
onnxMODEL = onnxruntime.InferenceSession("/content/tensorflow2onnx2.onnx" , providers=["CUDAExecutionProvider"] )
input_layer_name  = onnxMODEL.get_inputs()[0].name
output_layer_name = onnxMODEL.get_outputs()[0].name
print(input_layer_name)
print(output_layer_name)

# use onnx model

onnxinput  = np.random.normal(size=[1,224,224,3]).astype(np.float32)
start_time = time.time()
prediction = onnxMODEL.run(output_names= [output_layer_name], input_feed={input_layer_name : onnxinput })

end_time = time.time()
duration = end_time - start_time
print(f"Inference time with CUDA GPU : {duration} seconds")


keras_tensor_354
predictions
Inference time with CUDA GPU : 0.09988689422607422 seconds


# SCRFD

In [ ]:
onnxinput = torch.randn(1, 3, 640, 640)

onnx_model = onnxruntime.InferenceSession("models/scrfd_person_2.5g.onnx" , providers=["CPUExecutionProvider"])

for input_meta in onnx_model.get_inputs():
    print(f"Input name: {input_meta.name}, shape: {input_meta.shape}, type: {input_meta.type}")


input_name = onnx_model.get_inputs()[0].name
output_name = onnx_model.get_outputs()[0].name

#use onnx model
prediction = onnx_model.run(output_names=[output_name] , input_feed={input_name : onnxinput.detach().cpu().numpy() } )[0] # output_names= LIST , input_name= DICTIONARY{ input_name : input_value }
prediction

Input name: input.1, shape: [1, 3, '?', '?'], type: tensor(float)


array([[0.05280823],
       [0.02472493],
       [0.02263567],
       ...,
       [0.01731235],
       [0.02097017],
       [0.04348984]], dtype=float32)